# EnergyCast V1 — Step 8: Chronological Train / Validation / Test Split
## Electricity Price Forecasting System (Indian Electricity Market)

**Objective:** Implement strict temporal partitioning of the 28,176-hour engineered feature dataset into training, validation, and out-of-sample test splits.  
This notebook demonstrates why random cross-validation fails in time series and proves that our walk-forward chronological boundary preserves the arrow of time with zero data leakage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

SPLIT_DIR = Path('../data/processed/splits')
print('Split directory exists:', SPLIT_DIR.exists())

### 1. Load Chronologically Partitioned Datasets
We load `train.csv`, `val.csv`, and `test.csv` to verify temporal boundaries, record counts, and zero null values.

In [ ]:
train_df = pd.read_csv(SPLIT_DIR / 'train.csv')
val_df = pd.read_csv(SPLIT_DIR / 'val.csv')
test_df = pd.read_csv(SPLIT_DIR / 'test.csv')

train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])
val_df['timestamp'] = pd.to_datetime(val_df['timestamp'])
test_df['timestamp'] = pd.to_datetime(test_df['timestamp'])

total = len(train_df) + len(val_df) + len(test_df)
print(f'Train: {len(train_df):,} rows ({len(train_df)/total*100:.2f}%) | {train_df["timestamp"].min()} to {train_df["timestamp"].max()}')
print(f'Val:   {len(val_df):,} rows ({len(val_df)/total*100:.2f}%) | {val_df["timestamp"].min()} to {val_df["timestamp"].max()}')
print(f'Test:  {len(test_df):,} rows ({len(test_df)/total*100:.2f}%) | {test_df["timestamp"].min()} to {test_df["timestamp"].max()}')
print(f'Total: {total:,} rows across 60 features (Nulls: {train_df.isnull().sum().sum() + val_df.isnull().sum().sum() + test_df.isnull().sum().sum()})')

### 2. Comparative Target Statistics Across Splits
Evaluating the central tendency, dispersion, and price cap frequency across the splits.

In [ ]:
summary = pd.DataFrame([
    {'Split': 'Train (2022-2023)', 'Count': len(train_df), 'Mean': train_df['mcp_rs_per_mwh'].mean(), 'Median': train_df['mcp_rs_per_mwh'].median(), 'Std': train_df['mcp_rs_per_mwh'].std(), 'Cap_10k_Pct': (train_df['mcp_rs_per_mwh'] >= 10000).mean()*100},
    {'Split': 'Validation (2024)', 'Count': len(val_df), 'Mean': val_df['mcp_rs_per_mwh'].mean(), 'Median': val_df['mcp_rs_per_mwh'].median(), 'Std': val_df['mcp_rs_per_mwh'].std(), 'Cap_10k_Pct': (val_df['mcp_rs_per_mwh'] >= 10000).mean()*100},
    {'Split': 'Test (2025)', 'Count': len(test_df), 'Mean': test_df['mcp_rs_per_mwh'].mean(), 'Median': test_df['mcp_rs_per_mwh'].median(), 'Std': test_df['mcp_rs_per_mwh'].std(), 'Cap_10k_Pct': (test_df['mcp_rs_per_mwh'] >= 10000).mean()*100}
])
display(summary.round(2))

### 3. Chronological Timeline Partitioning Visualization
Visualizing the sequential walk-forward partitions without temporal overlap.

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(train_df['timestamp'], train_df['mcp_rs_per_mwh'], color='#1F4E79', alpha=0.5, lw=0.6, label='Train (2022-2023)')
plt.plot(val_df['timestamp'], val_df['mcp_rs_per_mwh'], color='#E67E22', alpha=0.6, lw=0.6, label='Validation (2024)')
plt.plot(test_df['timestamp'], test_df['mcp_rs_per_mwh'], color='#D9534F', alpha=0.7, lw=0.6, label='Test (2025)')

plt.axvline(train_df['timestamp'].max(), color='black', ls='--', lw=1.2, label='Train/Val Boundary')
plt.axvline(val_df['timestamp'].max(), color='black', ls=':', lw=1.2, label='Val/Test Boundary')
plt.title('Chronological Time-Series Partitioning (Arrow of Time Preserved)', fontweight='bold')
plt.ylabel('MCP (Rs/MWh)')
plt.legend(loc='upper right')
plt.show()

### 4. Diurnal Profile Stability Across Splits
Verifying that diurnal intraday peak dynamics (morning and evening peaks) remain consistent across all three partitions.

In [ ]:
h_tr = train_df.groupby(train_df['timestamp'].dt.hour)['mcp_rs_per_mwh'].mean()
h_va = val_df.groupby(val_df['timestamp'].dt.hour)['mcp_rs_per_mwh'].mean()
h_te = test_df.groupby(test_df['timestamp'].dt.hour)['mcp_rs_per_mwh'].mean()

plt.figure(figsize=(10, 5))
plt.plot(h_tr.index, h_tr.values, marker='o', color='#1F4E79', lw=2, label='Train Diurnal Profile')
plt.plot(h_va.index, h_va.values, marker='s', color='#E67E22', lw=2, label='Validation Diurnal Profile')
plt.plot(h_te.index, h_te.values, marker='^', color='#D9534F', lw=2, label='Test Diurnal Profile')
plt.title('Diurnal Hourly MCP Trajectory Across Partitions', fontweight='bold')
plt.xlabel('Hour of Day (0 to 23)')
plt.ylabel('Mean Price (Rs/MWh)')
plt.xticks(range(0, 24, 2))
plt.legend()
plt.show()

### 5. Summary & Hand-off to Model Building (Steps 9, 10, 11)
The data partitioning phase is complete. The datasets are ready for:
- **Step 9: Seasonal Naive Benchmark**
- **Step 10: Linear Regression / Ridge (Interpretable baseline)**
- **Step 11: Gradient Boosted Trees (LightGBM / XGBoost)**